In [ ]:
## LogisticRegression 하이퍼파라미터 공부
# 1. penalty — 정규화(regularization) 방식
##과적합을 막기 위해 계수(coefficient)에 페널티를 부여하는 방식
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=500, n_features=20, n_informative=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# L1: 희소(sparse)한 계수 -> 피처 선택 효과
model_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=1.0)
model_l1.fit(X_train, y_train)
print("0이 아닌 계수 개수 (L1):", (model_l1.coef_ != 0).sum())

# L2: 대부분 계수가 살아있지만 크기는 작아짐
model_l2 = LogisticRegression(penalty='l2', C=1.0)
model_l2.fit(X_train, y_train)
print("0이 아닌 계수 개수 (L2):", (model_l2.coef_ != 0).sum())



0이 아닌 계수 개수 (L1): 16
0이 아닌 계수 개수 (L2): 20


c:\Users\dsbang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\dsbang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\dsbang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default

In [ ]:
# 2. C — 정규화 강도 (역수)
## 가장 중요한 튜닝 대상입니다. C는 정규화 강도
import numpy as np

for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    coef_norm = np.linalg.norm(model.coef_)
    print(f"C={C:>6}: train={train_acc:.3f}, test={test_acc:.3f}, |coef|={coef_norm:.3f}")


C= 0.001: train=0.779, test=0.776, |coef|=0.186
C=  0.01: train=0.835, test=0.776, |coef|=0.703
C=   0.1: train=0.867, test=0.784, |coef|=1.611
C=     1: train=0.861, test=0.776, |coef|=2.217
C=    10: train=0.861, test=0.776, |coef|=2.342
C=   100: train=0.861, test=0.776, |coef|=2.357


# 3. solver — 최적화 알고리즘

내부적으로 손실함수를 최소화하는 알고리즘입니다. 데이터 크기, penalty 종류, 다중클래스 여부에 따라 선택이 달라집니다.

|solver	|지원 penalty	|특징|
|-------|--------------|----|
|'lbfgs' (기본값)	|l2, None	|중소규모 데이터에 무난, 빠름
|'liblinear'|	l1, l2	|소규모 데이터, 이진분류에 강함, OvR만 지원
|'saga'	l1, |l2, elasticnet, None	|대규모 데이터, 유일하게 elasticnet 지원
|'newton-cg'|	l2, None	|정밀하지만 느림
|'sag'	|l2, None	|대규모 데이터, feature scaling 필요

# 4. l1_ratio — L1/L2 혼합 비율 (elasticnet 전용)

penalty='elasticnet'일 때만 사용. 0~1 사이 값.

- l1_ratio=0 → 순수 L2
- l1_ratio=1 → 순수 L1
- l1_ratio=0.5 → 절반씩 혼합

# 5. class_weight — 클래스 불균형 처리

소수 클래스에 더 큰 가중치를 줘서 불균형 데이터를 보정합니다.

- None (기본값): 모든 클래스 동일 가중치
- 'balanced': n_samples / (n_classes * np.bincount(y)) 공식으로 자동 계산 → 소수 클래스에 큰 가중치
- {0: 1, 1: 5} 처럼 직접 지정 가능

# 6. max_iter — 최대 반복 횟수

경사하강법 등 반복 최적화의 최대 스텝 수. 기본값은 100.

- 너무 작으면 ConvergenceWarning과 함께 수렴 전에 학습 종료 → 성능 저하
- 피처가 많거나 스케일링이 안 된 데이터일수록 더 큰 값 필요

# 7. tol — 수렴 판정 허용오차

손실함수 개선폭이 이 값보다 작아지면 "수렴했다"고 보고 반복을 멈춥니다. 기본값 1e-4.

- tol이 작을수록 → 더 정밀하게 수렴할 때까지 반복 (느림, max_iter를 많이 씀)
- tol이 클수록 → 빨리 멈춤 (빠르지만 덜 정밀)

# 8. fit_intercept / intercept_scaling
- fit_intercept=True (기본): 절편(bias) 항을 학습. 대부분의 경우 True 유지
- intercept_scaling: solver='liblinear'이고 fit_intercept=True일 때, 절편에 대한 정규화 영향을 줄이기 위한 스케일 인자 (기본 1)

# 9. multi_class (구버전) / 자동 처리

최신 sklearn(1.5+)에서는 이 파라미터가 폐지 예정/제거되었고, solver가 자동으로 다항 로지스틱(multinomial)을 지원하면 그것을 쓰고, 아니면 OvR(One-vs-Rest)을 씁니다. lbfgs, saga, newton-cg 등은 multinomial을 지원하지만 liblinear는 OvR만 가능합니다. sklearn 버전에 따라 동작이 다르니 실제 사용 버전 문서를 확인하는 게 안전합니다.

# 10. random_state

solver가 'sag', 'saga', 'liblinear'처럼 확률적 요소를 포함할 때 재현성을 위해 고정합니다.

## 실전 튜닝 예제: GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),   # 로지스틱 회귀는 스케일링에 민감하므로 필수에 가까움
    ('clf', LogisticRegression(max_iter=5000, solver='saga'))
])

param_grid = [
    {'clf__penalty': ['l2'], 'clf__C': [0.01, 0.1, 1, 10, 100]},
    {'clf__penalty': ['l1'], 'clf__C': [0.01, 0.1, 1, 10, 100]},
    {'clf__penalty': ['elasticnet'], 'clf__C': [0.1, 1, 10], 'clf__l1_ratio': [0.2, 0.5, 0.8]},
]

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("최적 파라미터:", grid.best_params_)
print("최적 CV 점수:", grid.best_score_)
print("테스트 정확도:", grid.score(X_test, y_test))

# 핵심 요약
|우선순위 |	파라미터 |	튜닝 포인트 |
|-------|----------|------------|
|★★★	| C	|가장 먼저 튜닝. 로그 스케일(0.001~100)로 탐색
|★★★	| penalty + solver	|조합을 함께 결정
|★★	| class_weight	|클래스 불균형 있으면 필수 확인
|★★	|l1_ratio	|elasticnet 사용 시
|★	|max_iter, tol	|수렴 경고 뜰 때 조정